# NovaBank data preparation for fraud detection 

NovaBank is preparing to launch new fraud detection models and personalized product offers**. However, due to strict financial regulations, real customer data cannot be used. You work inside the analytics sandbox team. Your job is to work with a **synthetic data simulation**, ensure its quality, extract patterns, and build data assets for business stakeholders.

<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D01_NovaBank.png"/>

You received 3 CSV datasets from the simulation team:

* `customers.csv` – synthetic customer profiles
* `transactions.csv` – simulated financial transactions
* `loans.csv` – generated loan applications with approval outcomes

<Note type ="important">

The datasets have been generated with random data generation techniques. **They are available in the resources section of this lesson.**  Don't use the data generated from the first exercise.

The datasets have been generated with some intentional imperfections to mimic real-world data issues. 
</Note>

You must now **clean, transform, connect, and analyze** this data in preparation for use by:

* The **risk team** (who need high-risk flags and fraud summaries)
* The **product team** (who want to target segments by usage and behavior)
* The **executive team** (who want summary metrics for a quarterly report)

<Note type ="important">

In this exercise, be prepare to work on concepts that we will cover in future lessons, such as:
* Data aggregation and grouping
* Merging datasets

It's okay if you don't know how to do these things yet. The idea is get you thinking about the concepts and the types of questions you will be able to answer with your new skills.

</Note>


## Task 1: Data understanding

First, 

1. Load all three datasets into Pandas.
2. Print `.info()` and `.describe()` for each dataset.
3. Check data types. Convert columns if needed:
   * All dates (`signup_date`, `timestamp`, `application_date`) → datetime
   * All booleans (`is_active`, `is_fraud`, `approved`) → boolean
4. Are all `customer_id` values in `transactions_df` and `loans_df` also present in `customers_df`?
   
5. Answer these questions (with code):
   * How many rows in each dataset?
   * How many unique customers exist across all datasets?
   * Do any customers appear only in loans but not in transactions?
   * Are any duplicate `customer_id` or `transaction_id` values present?


<Note type="hint">

- Always check the size of each dataset right after loading. Use `.shape` to confirm row and column counts.

- Inspect data types before analysis. Convert dates with `pd.to_datetime()` and booleans with `.astype(bool)` to ensure proper filtering and comparisons.

- Use `set` logic to verify **referential integrity** between `customer_id` fields. Every `customer_id` in transactions and loans must exist in the customer table.

- Run `.duplicated().sum()` on ID fields to detect broken uniqueness, which can cause errors later in joins or grouping.

</Note>


In [63]:
import pandas as pd
import numpy as np

# chargement des datasets
customers_df = pd.read_csv('customers.csv')
transactions_df = pd.read_csv('transactions.csv')
loans_df = pd.read_csv('loans.csv')






In [64]:
# aperçu structurel des datasets
# print("Customers DataFrame Info:")
# customers_df.info()


# # aperçu statistique des datasets
# print("\nCustomers DataFrame Description:")
# print(customers_df.describe(include='all')) # include='all' pour inclure les colonnes non numériques

print("\nCustomers DataFrame Missing Values:")
print(customers_df.isnull().sum())


Customers DataFrame Missing Values:
customer_id     0
age             0
gender          0
country         0
account_type    0
signup_date     0
is_active       0
dtype: int64


In [65]:
customers_df.head()

,customer_id,age,gender,country,account_type,signup_date,is_active
0,CUST_000001,55,Male,Germany,SAVINGS,2021-11-10,True
1,CUST_000002,43,Male,Germany,checking,2022-12-08,True
2,CUST_000003,38,Male,France,savings,2020-05-25,True
3,CUST_000004,39,female,Italy,savings,2021-04-11,True
4,CUST_000005,29,Female,France,checking,2022-10-01,True


In [66]:
print("\nTransactions DataFrame Info:")
transactions_df.info()
print("\nTransactions DataFrame Description:")
print(transactions_df.describe(include='all'))


Transactions DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25010 entries, 0 to 25009
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    25010 non-null  object 
 1   customer_id       25010 non-null  object 
 2   amount            25010 non-null  float64
 3   transaction_type  25010 non-null  object 
 4   timestamp         25010 non-null  object 
 5   is_fraud          25010 non-null  bool   
dtypes: bool(1), float64(1), object(4)
memory usage: 1001.5+ KB

Transactions DataFrame Description:
                              transaction_id  customer_id        amount  \
count                                  25010        25010  25010.000000   
unique                                 25000         2474           NaN   
top     43548ecd-de09-4364-b4fd-d276bfd69168  CUST_000501           NaN   
freq                                       2           25           NaN   
mean      

In [67]:
transactions_df.head()

,transaction_id,customer_id,amount,transaction_type,timestamp,is_fraud
0,99466798-e548-438c-98f5-8cb3653787ac,CUST_001804,6.94,transfer,2022-06-20 09:02:12,False
1,43b7620b-cc24-41be-b369-8f6d57554c72,CUST_000215,4.95,purchase,2022-10-17 13:47:39,False
2,9972fbb3-13e7-43f7-b33b-ed614a88b172,CUST_001677,100.00,withdrawal,2023-12-18 10:28:17,False
3,db183eb7-7dfa-4dae-afef-3862df4375e7,CUST_001283,10.07,refund,2023-05-21 05:06:09,False
4,7d0ce8fa-e565-4c8a-b6bd-01fbf424c8fa,CUST_000312,200.00,withdrawal,2023-12-31 02:26:44,False


In [68]:
print("\nLoans DataFrame Info:")
loans_df.info()
print("\nLoans DataFrame Description:")     
print(loans_df.describe(include='all'))


Loans DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   loan_id           3000 non-null   object 
 1   customer_id       3000 non-null   object 
 2   loan_type         3000 non-null   object 
 3   loan_amount       3000 non-null   float64
 4   repayment_term    3000 non-null   int64  
 5   application_date  3000 non-null   object 
 6   approved          3000 non-null   bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 143.7+ KB

Loans DataFrame Description:
            loan_id  customer_id loan_type    loan_amount  repayment_term  \
count          3000         3000      3000    3000.000000     3000.000000   
unique         3000         1132         5            NaN             NaN   
top     LOAN_000001  CUST_000191  personal            NaN             NaN   
freq              1            8     

In [ ]:
loans_df.head()

,loan_id,customer_id,loan_type,loan_amount,repayment_term,application_date,approved
0,LOAN_000001,CUST_000511,mortgage,264370.39,15,2023-12-21,False
1,LOAN_000002,CUST_002000,mortgage,216546.35,20,2023-01-14,True
2,LOAN_000003,CUST_001690,business,10000.00,3,2021-08-24,True
3,LOAN_000004,CUST_000501,mortgage,208070.09,30,2021-04-14,False
4,LOAN_000005,CUST_001196,personal,8542.61,2,2021-02-28,True


In [70]:
# conversion des colonnes de date en type datetime
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'], errors='coerce') 
# errors='coerce' pour gérer les valeurs non convertibles en NaT sans lever d'erreur. 
# utile pour filter ou imputer les dates manquantes plus tard.
transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'], errors='coerce')
loans_df['application_date'] = pd.to_datetime(loans_df['application_date'], errors='coerce')

In [71]:
# conversion des booléens
for col in ['is_active', 'is_fraud', 'approved']:
    for df in [customers_df, transactions_df, loans_df]:
        if col in df.columns: # columns permet de vérifier si la colonne existe dans le DataFrame
            df[col] = df[col].astype(bool)

# 1.5

In [72]:
# nombre de lignes dans chaque dataset
print(f"\nNumber of rows in Customers DataFrame: {customers_df.shape[0]} rows, {customers_df.shape[1]} columns")
print(f"Number of rows in Transactions DataFrame: {transactions_df.shape[0]} rows, {transactions_df.shape[1]} columns")
print(f"Number of rows in Loans DataFrame: {loans_df.shape[0]} rows, {loans_df.shape[1]} columns")



Number of rows in Customers DataFrame: 2005 rows, 7 columns
Number of rows in Transactions DataFrame: 25010 rows, 6 columns
Number of rows in Loans DataFrame: 3000 rows, 7 columns


In [73]:
# nombre de clients uniques dans chaque dataset
print(f"\nNumber of unique customers in Customers DataFrame: {customers_df['customer_id'].nunique()}") 
# ou customers_df.customer_id.unique()
print(f"Number of unique customers in Transactions DataFrame: {transactions_df['customer_id'].nunique()}")
print(f"Number of unique customers in Loans DataFrame: {loans_df['customer_id'].nunique()}")



Number of unique customers in Customers DataFrame: 2000
Number of unique customers in Transactions DataFrame: 2474
Number of unique customers in Loans DataFrame: 1132


In [74]:
# nombre de clients apparaissant uniquement dans loans mais pas transactions
check_inintegrite_ref=len((set(loans_df['customer_id']) - set(transactions_df['customer_id'])))
check_inintegrite_ref

34

In [75]:
# vérifions si ces clients existent dans customers
clients_inexistants=len((set(loans_df['customer_id']))  - set(customers_df['customer_id']))
clients_inexistants

34

In [76]:
check_inintegrite_ref2 = len(set(transactions_df['customer_id']) - set(loans_df['customer_id']))
check_inintegrite_ref2

1376

In [61]:
clients_inexistants2= len(set(transactions_df['customer_id']) - set(customers_df['customer_id']))
clients_inexistants2

474

In [77]:
transactions_df.customer_id.value_counts().head()


customer_id
CUST_000501    25
CUST_000681    25
CUST_000387    25
CUST_001818    25
CUST_000606    25
Name: count, dtype: int64

In [ ]:
# vérifications des doublons dans chaque dataset
print(f"\nNumber of duplicate rows in Customers DataFrame: {customers_df.customer_id.duplicated().sum()}")
print(f"Number of duplicate rows in Transactions DataFrame: {transactions_df.transaction_id.duplicated().sum()}")
print(f"Number of duplicate rows in Loans DataFrame: {loans_df.loan_id.duplicated().sum()}")


Number of duplicate rows in Customers DataFrame: 5
Number of duplicate rows in Transactions DataFrame: 10
Number of duplicate rows in Loans DataFrame: 0


## Task 2: Data cleaning and standardization 

The simulation team warns you that some data quality rules were intentionally relaxed to reflect real-world chaos. You must now clean and standardize before doing any analysis.

1. Drop duplicates across all datasets.
2. Remove transactions where `amount <= 0` or unusually high (over €50,000).
3. Fix formatting in:
   * `gender` values (make consistent case and spelling)
   * `account_type` (convert to lowercase, strip spaces)
   * `country` (use `.value_counts()` to find unexpected values)

4. Ensure column names use `snake_case` throughout.
5. Add a column to `customers_df`, `customer_age_group` with "18–25", "26–40", "41–60", "60+" using bins.

6. Add a new column to `transactions_df`, `day_of_week` from the timestamp.


<Note type="hint">

- Always remove duplicates early to avoid inflated counts or misleading summaries. Use `.drop_duplicates()` on each table.
- Filter out invalid transactions—remove rows where amounts are zero, negative, or unrealistically large (e.g. over €50,000).
- Standardize inconsistent text values (like gender or account type) using `.replace()` or `.str.lower().str.strip()`.
- Use `pd.cut()` to group ages into ranges for better segmentation, and extract weekday names with `.dt.day_name()` to analyze behavior over time.

</Note>


In [26]:
# suppression des doublons
customers_df = customers_df.drop_duplicates(subset='customer_id')
transactions_df = transactions_df.drop_duplicates(subset='customer_id') 
loans_df = loans_df.drop_duplicates(subset='customer_id')

In [27]:
# suppression des transactions anormales (montant négatif ou supérieur à 50 000)
transactions_df = transactions_df[(transactions_df['amount'] > 0) & (transactions_df['amount'] <= 50000)]

In [ ]:
customers_df.groupby("gender")["gender"].count()

In [28]:
# mise en forme
customers_df['gender'] = customers_df['gender'].str.lower().str.strip() 
# str 2 fois parce que str.lower() renvoie une Series, donc on peut chaîner str.strip() dessus.
customers_df['gender'] = customers_df['gender'].replace({'femal': 'female', 'maleee': 'male', "f": "female", "m": "male", "fem": "female", "masc": "male"}) 
# chercher comment utiliser un regex ici 

In [30]:
# mise en forme type de compte
customers_df['account_type'] = customers_df['account_type'].str.lower().str.strip()
print(f"Valeurs inattendues : {customers_df['country'].value_counts()}")


Valeurs inattendues : country
Germany        492
France         415
Spain          299
Italy          235
Netherlands    198
Belgium        154
Austria        118
Portugal        89
Name: count, dtype: int64


# 2.4

In [31]:
# utiliser snake_case pour tous les noms de colonnes
customers_df['customers_age_group']= pd.cut(customers_df['age'], bins=[18, 25, 40, 60, 120], labels=['18-25', '26-40', '41-60', '60+'], right=True)
# right true pour inclure la borne droite dans l'intervalle

# 2.6

In [32]:
transactions_df['day_of_week'] = transactions_df['timestamp'].dt.day_name()

## Task 3: Customer summary and scoring
The executive team wants a high-level summary of customer activity and a scoring system to identify high-value customers.

1. Calculate for each customer:

   * Total number of transactions
   * Total transaction amount
   * Average transaction amount
   * Most common transaction type

2. Build a new DataFrame `customer_summary_df` with:

   * `customer_id`
   * `total_transactions`
   * `total_amount`
   * `average_amount`
   * `most_common_txn_type`

3. Merge `customer_summary_df` with `customers_df`.

4. Create a `customer_value_score`:

   * Formula: `(total_amount × is_active) ÷ age`
   * This gives more weight to younger active customers who spend more.

5. Sort and print top 20 customers by value score.


<Note type="hint">

- Use `.groupby()` to calculate metrics per customer, like transaction count, total spend, and average amount. Use `.mode()` to find the most common transaction type.
- Always `.reset_index()` after aggregation to keep the table flat and merge-ready.
- Merge your summary with the original customer table to add age and activity status with this command : 
`customer_summary_df = customer_summary.merge(customers_df, on='customer_id', how='left')`
- Handle missing values before scoring—set `is_active` to 0 for inactive or missing users with `fillna(0)`
- Create a score by combining spend and activity, then normalize by age.
- Use `.nlargest()` to rank your most valuable customers.

</Note>

In [ ]:
txn_summary = transactions_df.groupby("customer_id").agg(
    # nbre de transactions non nulles pour ce client
    total_transactions=("transaction_id", 'count'), 
    # créer une nouvelle colonne total_transactions qui compte le nombre de 
    # transaction_id par customer_id en appliquant la fonction d'agrégation 'count'
    total_amount=("amount", 'sum'),
    average_amount=("amount", 'mean'),  
    most_common_transaction = ("amount", lambda x: x.mode()[0] 
    # lambda fonction anonyme pour appliquer mode() sur chaque groupe
    # en cas de multiple modes, on prend juste la première valeur avec [0]
    # if not x.mode().empty else np.nan à gérer dans un try except
    # pour éviter les erreurs systèmes si mode() est vide
    )).reset_index()
# reset_index() pour remettre customer_id comme colonne normale après le groupby
txn_summary

,customer_id,total_transactions,total_amount,average_amount,most_common_transaction
0,CUST_000001,1,27.54,27.54,27.54
1,CUST_000002,1,111.50,111.50,111.50
2,CUST_000003,1,36.29,36.29,36.29
3,CUST_000004,1,9.24,9.24,9.24
4,CUST_000005,1,403.48,403.48,403.48
...,...,...,...,...,...
2465,CUST_099305,1,51.05,51.05,51.05
2466,CUST_099531,1,190.23,190.23,190.23
2467,CUST_099736,1,50.00,50.00,50.00
2468,CUST_099861,1,200.00,200.00,200.00


In [40]:
# new data frame customer_summary
customer_summary_df = txn_summary.merge(customers_df, on='customer_id', how='left')
# how='left' pour garder tous les clients même ceux sans transactions
# mais ajoute des valeurs NaN pour les colonnes de txn_summary si pas de correspondance
# avec customers_df d'où :
customer_summary_df['is_active'].isna().sum()

473

In [41]:
customer_summary_df["is_active"] = customer_summary_df["is_active"].fillna(False)
# on remplace les NaN par False pour indiquer que ces clients ne sont pas actifs

C:\Users\elomo\AppData\Local\Temp\ipykernel_59552\2853142555.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  customer_summary_df["is_active"] = customer_summary_df["is_active"].fillna(False)


# 3.4 

In [ ]:
# create a customer value score
customer_summary_df['customer_value_score'] = (customer_summary_df.total_amount * customer_summary_df.is_active.astype(int)) / customer_summary_df.age
customer_summary_df.sort_values(by='customer_value_score', ascending=False).head(20).snake_case()
# top_20 = customer_summary_df.nlargest(20, "customer_value_score")
# top_20


,customer_id,total_transactions,total_amount,average_amount,most_common_transaction,age,gender,country,account_type,signup_date,is_active,customers_age_group,customer_value_score
1589,CUST_001593,1,11788.75,11788.75,11788.75,66.0,female,Spain,checking,2022-04-12,True,60+,178.617424
1437,CUST_001441,1,3241.10,3241.10,3241.10,27.0,male,Austria,business,2023-08-22,True,26-40,120.040741
1399,CUST_001403,1,1604.77,1604.77,1604.77,26.0,female,Belgium,savings,2020-10-16,True,26-40,61.721923
653,CUST_000655,1,1395.54,1395.54,1395.54,23.0,female,Netherlands,savings,2020-10-02,True,18-25,60.675652
1307,CUST_001310,1,3260.58,3260.58,3260.58,56.0,female,Germany,business,2022-12-01,True,41-60,58.224643
1120,CUST_001122,1,2129.44,2129.44,2129.44,38.0,female,Germany,checking,2023-09-12,True,26-40,56.037895
404,CUST_000406,1,1686.32,1686.32,1686.32,34.0,male,Netherlands,savings,2019-11-10,True,26-40,49.597647
1383,CUST_001386,1,1689.23,1689.23,1689.23,40.0,male,Netherlands,savings,2022-06-12,True,26-40,42.230750
1610,CUST_001614,1,1335.70,1335.70,1335.70,35.0,male,France,checking,2022-04-22,True,26-40,38.162857
360,CUST_000362,1,682.33,682.33,682.33,18.0,male,France,premium,2020-11-11,True,NaN,37.907222


## Task 4: Fraud pattern analysis

Then the risk team wants insights into fraudulent transactions to help refine their detection models.

1. Calculate:

   * Total number of fraudulent transactions
   * % fraud by transaction type
   * Fraud distribution by day of the week

2. Identify:

   * Top 10 customers with the most fraud
   * % of fraud by country
   * Any customers with more than **2** fraud cases

3. Add a `fraud_rate` column to `customer_summary_df`:

   * Defined as: (fraud\_count / total\_transactions)

4. Create a flag:

   * `potential_risk` = `True` if fraud\_rate > 0.1 and account\_type is not “business”



<Note type="hint">

- Use `.sum()` and `.mean()` to measure total fraud and fraud rate.
- Group by transaction type or weekday to detect high-risk patterns.
- Merge transactions with customer info to analyze fraud by country.
- Use `.groupby('customer_id')` to identify accounts with frequent fraud.
- Flag customers with a high personal fraud rate (e.g., over 10%) especially if they’re not business accounts.

</Note>


## Task 5: Analyse loan behavior

The **product team** needs insight into who’s applying, getting rejected, and what factors affect approval.

1. Group loans by:

   * `loan_type` → Get average amount, approval rate, and repayment term
   * `age_group` → Popular loan types by age
   * `country` → Approval rate by country

2. Merge `loans_df` with `customers_df` for richer analysis.

3. Add flag `loan_rejected` → True where `approved == False`

4. Create a new column:

   * `loan_to_income_ratio` → (loan\_amount / total\_transaction\_amount)
   * Handle divide-by-zero cases with `np.where()` or `.apply()` safely

5. Who has the highest ratio? Do rejected applicants have higher ratios?

<Note type="hint">

- Group loans by type to compare average amount, approval rate, and term length.
- Merge loan data with customer profiles to explore age and country effects. Use `.pivot()` to compare loan demand by age group, and `.mean()` to calculate approval rates.
- Add a rejection flag using `~df['approved']` to quickly measure rejections.
- To measure risk, calculate **loan-to-income ratio** by dividing loan amount by a customer’s total transactions. Use `np.where()` to avoid dividing by zero.
- Compare these ratios for approved vs. rejected loans to uncover behavioral patterns.

</Note>

## Task 6: Export data assets

Finally, export the cleaned and enriched datasets for each team.
1. Export `customer_summary_df` to `customer_summary.csv` for the executive team.
2. Export `transactions_df` with fraud flags to `transactions_fraud_analysis.csv` for the risk team.
3. Export `loans_df` with enrichment to `loans_product_analysis.csv` for the product team.
4. Ensure all exports have no index column and use UTF-8 encoding.
5. Print the first 5 rows of each exported DataFrame to confirm.

<Note type="hint">

- Use `.to_csv('filename.csv', index=False, encoding='utf-8')` to export without index and ensure proper character encoding.
- Always preview the first few rows of your exported DataFrames with `.head()` to confirm the data looks correct before sharing.
  
</Note>
